# Assignment 3: Policy-Gradient and Actor-Critic Methods

## Instructions and marking

There are exactly two questions. Each question has two code parts and one inference response.

### You may modify only

- The code bodies in the four cells marked **Q1A/Q1B/Q2A/Q2B**.
- The two Markdown cells marked **Write your answer here**.

### You must not modify

- Function or class names, argument order, default values, or return formats.
- Imports, provided classes, OR-Gym environment factories, rollout/replay helpers, random seeds, or cell tags.
- The public-test notebook or the expected submission filename.
- OR-Gym source code or environment transition logic.

### Randomness

Call `set_seed(seed)` whenever a provided driver or test supplies a seed. Do not create an unseeded random generator. Private tests use different seeds, rollout lengths, batch sizes, state dimensions, and action bounds.

In [ ]:
# Run this cell once only if the packages are not already installed.
# OR-Gym 0.5.0 declares an old Gym dependency, so Gym is installed separately.
%pip install -q gym==0.26.2 scipy pandas matplotlib networkx
%pip install -q --no-deps or-gym==0.5.0

In [ ]:
import copy
import json
import random
from collections import deque
from pathlib import Path

import numpy as np

# Compatibility aliases required by OR-Gym 0.5.0 with NumPy 2.x.
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_
if not hasattr(np, "Inf"):
    np.Inf = np.inf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

# These are the actual OR-Gym package environments behind Knapsack-v3 and PortfolioOpt-v0.
from or_gym.envs.classic_or.knapsack import OnlineKnapsackEnv
from or_gym.envs.finance.portfolio_opt import PortfolioOptEnv


def set_seed(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch with the same integer."""
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))


def make_knapsack_env(seed: int = 0):
    """Create the OR-Gym Knapsack-v3 environment with its 4-vector state."""
    set_seed(seed)
    return OnlineKnapsackEnv(env_config={"mask": False, "seed": int(seed)})


def make_portfolio_env(seed: int = 0):
    """Create and seed the OR-Gym PortfolioOpt-v0 environment."""
    set_seed(seed)
    env = PortfolioOptEnv()
    env.seed(int(seed))
    env.reset()
    return env


class ActorCritic(nn.Module):
    """Provided shared-body actor-critic network used by each A3C worker."""

    def __init__(self, state_dim: int, n_actions: int, hidden_dim: int = 64):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(state_dim, hidden_dim), nn.Tanh())
        self.policy_head = nn.Linear(hidden_dim, n_actions)
        self.value_head = nn.Linear(hidden_dim, 1)

    def forward(self, states: torch.Tensor):
        h = self.shared(states.float())
        return self.policy_head(h), self.value_head(h).squeeze(-1)


@torch.no_grad()
def collect_a3c_rollout(env, model, state, t_max: int = 5):
    """Provided OR-Gym rollout helper. It does not update any parameters."""
    states, actions, rewards, dones = [], [], [], []
    current = np.asarray(state, dtype=np.float32)
    done = False
    for _ in range(int(t_max)):
        state_t = torch.as_tensor(current, dtype=torch.float32).unsqueeze(0)
        logits, _ = model(state_t)
        action = int(Categorical(logits=logits).sample().item())
        next_state, reward, done, _ = env.step(action)
        states.append(current.copy())
        actions.append(action)
        rewards.append(float(reward))
        dones.append(float(done))
        current = np.asarray(next_state, dtype=np.float32)
        if done:
            break

    if done:
        bootstrap_value = torch.tensor(0.0, dtype=torch.float32)
    else:
        _, next_value = model(torch.as_tensor(current).float().unsqueeze(0))
        bootstrap_value = next_value.squeeze(0).detach()

    rollout = {
        "states": torch.as_tensor(np.asarray(states), dtype=torch.float32),
        "actions": torch.as_tensor(actions, dtype=torch.long),
        "rewards": torch.as_tensor(rewards, dtype=torch.float32),
        "dones": torch.as_tensor(dones, dtype=torch.float32),
        "bootstrap_value": bootstrap_value,
    }
    return rollout, current, done


class ReplayBuffer:
    """Provided replay buffer for DDPG. Students do not modify this class."""

    def __init__(self, capacity: int):
        self.data = deque(maxlen=int(capacity))

    def __len__(self):
        return len(self.data)

    def add(self, state, action, reward, next_state, done):
        self.data.append(
            (
                np.asarray(state, dtype=np.float32).copy(),
                np.asarray(action, dtype=np.float32).copy(),
                float(reward),
                np.asarray(next_state, dtype=np.float32).copy(),
                float(done),
            )
        )

    def sample(self, batch_size: int, seed: int):
        rng = np.random.default_rng(int(seed))
        indices = rng.choice(len(self.data), size=int(batch_size), replace=False)
        rows = [self.data[int(i)] for i in indices]
        states, actions, rewards, next_states, dones = map(np.asarray, zip(*rows))
        return {
            "states": torch.as_tensor(states, dtype=torch.float32),
            "actions": torch.as_tensor(actions, dtype=torch.float32),
            "rewards": torch.as_tensor(rewards, dtype=torch.float32).view(-1, 1),
            "next_states": torch.as_tensor(next_states, dtype=torch.float32),
            "dones": torch.as_tensor(dones, dtype=torch.float32).view(-1, 1),
        }

---

## Question 1: Standard A3C on OR-Gym `Knapsack-v3`

`Knapsack-v3` presents one item at a time. Action `0` rejects the item and action `1` accepts it. This is a discrete control problem, so the actor represents the stochastic policy $\pi_\theta(a\mid s)$ using a categorical distribution. The critic represents $V_w(s)$.


### Q1A: n-step return and A3C loss

Implement both functions in the next cell.

One-step TD advantage:

$$\delta_t = R_{t+1}+\gamma V_w(S_{t+1})-V_w(S_t).$$

A3C extends the same bootstrapping idea to an n-step rollout. `compute_n_step_returns(rewards, dones, bootstrap_value, gamma)` must return one `torch.float32` tensor of shape `(T,)`, on the same device as `rewards`. Starting with $R_T=$ `bootstrap_value`, compute backwards for $t=T-1,\ldots,0$:

$$R_t=\text{rewards}[t]+\gamma(1-\text{dones}[t])R_{t+1}.$$

If `dones[t] == 1`, the return at that transition must not bootstrap. Do not modify any input tensor.

`a3c_loss(logits, values, actions, returns, value_coef, entropy_coef)` uses the n-step advantage $\hat A_t=R_t-V_w(S_t)$ and must return exactly four scalar tensors in this order:

1. `total_loss = policy_loss + value_coef * value_loss - entropy_coef * entropy`
2. `policy_loss = mean(-log pi(a_t|s_t) * stop_gradient(returns_t - values_t))`
3. `value_loss = 0.5 * mean((returns_t - values_t)^2)`
4. `entropy = mean entropy of Categorical(logits=logits)`

The input shapes are `logits: (T, A)`, `values: (T,)`, `actions: (T,)`, and `returns: (T,)`. Detach the advantage only in the actor loss. The value loss must still update the critic.

### Q1B: one asynchronous worker-to-global update

`a3c_worker_update(...)` must perform exactly one worker update in this order:

1. Load `global_model.state_dict()` into `local_model`.
2. Run the local actor and critic on `rollout["states"]`.
3. Use Q1A to compute the n-step returns and the A3C loss.
4. Clear gradients on both `global_optimizer` and `local_model`.
5. Backpropagate through `local_model` and clip its gradient norm to `max_grad_norm`.
6. Copy every non-`None` local gradient to the matching global parameter.
7. Call `global_optimizer.step()` exactly once and reload the updated global parameters into the local model.

Return a Python `dict` with exactly the keys `total_loss`, `policy_loss`, `value_loss`, `entropy`, and `grad_norm`. Each value must be a finite Python `float` measured before the optimizer step. Do not return a model, tensor, or optimizer.

In [ ]:
# Q1A. Modify only the two function bodies in this cell.
def compute_n_step_returns(
    rewards: torch.Tensor,
    dones: torch.Tensor,
    bootstrap_value: torch.Tensor,
    gamma: float,
) -> torch.Tensor:
    # 1. Allocate a float32 tensor with T entries on rewards.device.
    # 2. Start the running return from a detached scalar bootstrap value.
    # 3. Move backwards through the rollout using the stated terminal mask.
    # 4. Return shape (T,); do not modify rewards or dones.
    raise NotImplementedError


def a3c_loss(
    logits: torch.Tensor,
    values: torch.Tensor,
    actions: torch.Tensor,
    returns: torch.Tensor,
    value_coef: float = 0.5,
    entropy_coef: float = 0.01,
):
    # 1. Construct Categorical(logits=logits) and obtain log pi(a_t|s_t).
    # 2. Compute the n-step advantages returns - values.
    # 3. Detach the advantages only in the policy loss.
    # 4. Compute value loss, mean entropy, and total loss exactly as specified.
    # 5. Return (total_loss, policy_loss, value_loss, entropy), all scalars.
    raise NotImplementedError

In [ ]:
# Q1B. Modify only this function body.
def a3c_worker_update(
    global_model: nn.Module,
    local_model: nn.Module,
    global_optimizer: torch.optim.Optimizer,
    rollout: dict,
    gamma: float = 0.99,
    value_coef: float = 0.5,
    entropy_coef: float = 0.01,
    max_grad_norm: float = 40.0,
) -> dict:
    # Follow the seven worker-to-global steps in Q1B in the stated order.
    # The optimizer owns global parameters; gradients are computed locally.
    # Copy detached clones of all non-None local gradients to global parameters.
    # Return exactly the five specified finite Python floats.
    raise NotImplementedError

### Q1)

Explain why asynchronous workers and n-step rollouts can be useful for the stochastic online knapsack task. Limit your answer to 120 words.

Write your answer here.

---

## Question 2: DDPG on OR-Gym `PortfolioOpt-v0`
`PortfolioOpt-v0` has a 7-dimensional state and a 3-dimensional continuous action. Each action contains the number of shares to buy or sell for the three assets and is bounded component-wise by the environment.

The replay buffer is provided. Exploration-noise generation and the long training loop are also provided by the platform and are not graded. You implement the bounded actor, critic, TD target, actor/critic optimizer steps, and target-network update.

### Q2A: bounded deterministic actor and action-value critic

Implement the two classes in the next cell.

`Actor(state_dim, action_low, action_high, hidden_dim)` must use exactly

`Linear(state_dim, hidden_dim) -> ReLU -> Linear(hidden_dim, hidden_dim) -> ReLU -> Linear(hidden_dim, action_dim) -> Tanh`.

Register `action_low` and `action_high` as one-dimensional `torch.float32` buffers. For `states` of shape `(B, state_dim)`, return a `torch.float32` tensor of shape `(B, action_dim)` using

`centre + half_range * tanh_output`,

where `centre = (action_high + action_low) / 2` and `half_range = (action_high - action_low) / 2`. Every action component must lie within its supplied lower and upper bounds.

`Critic(state_dim, action_dim, hidden_dim)` must concatenate states and actions along dimension `1` and use exactly

`Linear(state_dim + action_dim, hidden_dim) -> ReLU -> Linear(hidden_dim, hidden_dim) -> ReLU -> Linear(hidden_dim, 1)`.

For inputs with shapes `(B, state_dim)` and `(B, action_dim)`, return a `torch.float32` tensor of shape `(B, 1)` containing one estimated $Q_w(s,a)$ per row.

### Q2B: DDPG TD target and parameter updates

`soft_update(target, source, tau)` must update every target parameter in place under `torch.no_grad()`:

$$\text{target}\leftarrow(1-\tau)\,\text{target}+\tau\,\text{source}.$$

It must return `None`.

`ddpg_update(...)` receives a replay batch with exactly the keys `states`, `actions`, `rewards`, `next_states`, and `dones`. `rewards` and `dones` have shape `(B, 1)`. Perform exactly these steps:

1. Under `torch.no_grad()`, compute `next_actions = target_actor(next_states)`.
2. Compute the one-step bootstrapped target $y=r+\gamma(1-d)Q_{w^-}(s',\mu_{\theta^-}(s'))$.
3. Update the critic exactly once using the mean-squared error between `critic(states, actions)` and the target.
4. Update the actor exactly once using `actor_loss = -mean(critic(states, actor(states)))`.
5. Soft-update `target_actor` from `actor` and `target_critic` from `critic` exactly once.

Return a Python `dict` with exactly the keys `critic_loss`, `actor_loss`, and `mean_target_q`. Each value must be a finite Python `float` from that update. Do not return a model, tensor, optimizer, or replay batch.

In [ ]:
# Q2A. Modify only the two class bodies in this cell.
class Actor(nn.Module):
    def __init__(self, state_dim: int, action_low, action_high, hidden_dim: int = 128):
        super().__init__()
        # 1. Convert both bounds to 1-D float32 tensors and register them as buffers.
        # 2. Build the exact deterministic-actor network specified in Q2A.
        raise NotImplementedError

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        # 1. Compute the Tanh network output.
        # 2. Scale it component-wise from [-1, 1] to [action_low, action_high].
        # 3. Return float32 values with shape (B, action_dim).
        raise NotImplementedError


class Critic(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super().__init__()
        # Build the exact Q(s,a) network specified in Q2A.
        raise NotImplementedError

    def forward(self, states: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
        # Concatenate along dim=1 and return shape (B, 1), one Q-value per row.
        raise NotImplementedError

In [ ]:
# Q2B. Modify only the two function bodies in this cell.
def soft_update(target: nn.Module, source: nn.Module, tau: float) -> None:
    # Apply the stated in-place update under torch.no_grad(); return None.
    raise NotImplementedError


def ddpg_update(
    actor: nn.Module,
    critic: nn.Module,
    target_actor: nn.Module,
    target_critic: nn.Module,
    actor_optimizer: torch.optim.Optimizer,
    critic_optimizer: torch.optim.Optimizer,
    batch: dict,
    gamma: float = 0.99,
    tau: float = 0.005,
) -> dict:
    # Follow the five DDPG update steps in Q2B in the stated order.
    # Build no autograd graph for the one-step TD target.
    # Each optimizer must take exactly one step.
    # Return exactly the three specified finite Python floats.
    raise NotImplementedError

### Q2) 

A Gaussian stochastic policy is used for continuous actions, whereas DDPG uses a deterministic actor. Explain this difference for the portfolio task and state how the action-value critic provides a learning signal to the DDPG actor.

Write your answer here.